# Colab Notebook Orchestrator

Use this notebook to run notebooks from `src/` in a controlled order.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os
import sys

PROJECT_DIR = Path('/content/drive/MyDrive/GNN')
SRC_DIR = PROJECT_DIR / 'src'
DATA_DIR = PROJECT_DIR / 'data'
OUTPUT_DIR = PROJECT_DIR / 'output'
EXECUTED_DIR = OUTPUT_DIR / 'executed_notebooks'
LOG_DIR = OUTPUT_DIR / 'logs'

for path in [OUTPUT_DIR, EXECUTED_DIR, LOG_DIR, OUTPUT_DIR / 'models', OUTPUT_DIR / 'statis', OUTPUT_DIR / 'picts']:
    path.mkdir(parents=True, exist_ok=True)

os.chdir(SRC_DIR)
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print('PROJECT_DIR =', PROJECT_DIR)
print('SRC_DIR     =', SRC_DIR)
print('DATA_DIR    =', DATA_DIR)
print('OUTPUT_DIR  =', OUTPUT_DIR)


In [ ]:
!pip install -q nbconvert torch-geometric optuna pyxdameraulevenshtein nltk


## Choose Execution Order

Edit `RUN_NOTEBOOKS` to control which notebooks run and in what order.

In [ ]:
AVAILABLE_NOTEBOOKS = [
    'DataProcess.ipynb',
    'GatConvCall.ipynb',
    'GatConvStatusEmbCall.ipynb',
    'GatConvTimeDecayCall.ipynb',
    'GatConvTimeDecayStatusEmbCall.ipynb',
    'PrefixEmbeddingGCNCall.ipynb',
]

PIPELINES = {
    'data_only': [
        'DataProcess.ipynb',
    ],
    'baseline_gat': [
        'GatConvCall.ipynb',
    ],
    'status_edge_gat': [
        'GatConvStatusEmbCall.ipynb',
    ],
    'time_decay_gat': [
        'GatConvTimeDecayCall.ipynb',
    ],
    'time_decay_status_edge_gat': [
        'GatConvTimeDecayStatusEmbCall.ipynb',
    ],
    'prefix_embedding_gcn': [
        'PrefixEmbeddingGCNCall.ipynb',
    ],
    'all_gat_models': [
        'GatConvCall.ipynb',
        'GatConvStatusEmbCall.ipynb',
        'GatConvTimeDecayCall.ipynb',
        'GatConvTimeDecayStatusEmbCall.ipynb',
    ],
}

# Default: run the current main training notebook only.
# Change this to PIPELINES['all_gat_models'] or write your own list.
RUN_NOTEBOOKS = PIPELINES['status_edge_gat']

# If True, stop immediately when one notebook fails.
# If False, continue running the remaining notebooks and summarize failures at the end.
STOP_ON_ERROR = True

missing = [name for name in RUN_NOTEBOOKS if not (SRC_DIR / name).exists()]
if missing:
    raise FileNotFoundError(f'Missing notebooks in src/: {missing}')

print('Execution order:')
for idx, notebook_name in enumerate(RUN_NOTEBOOKS, start=1):
    print(f'{idx}. {notebook_name}')


In [ ]:
import json
import subprocess
import time
from datetime import datetime

RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S')

def execute_notebook(notebook_name):
    input_path = SRC_DIR / notebook_name
    output_name = f"{Path(notebook_name).stem}_executed_{RUN_ID}.ipynb"
    output_path = EXECUTED_DIR / output_name
    log_path = LOG_DIR / f"{Path(notebook_name).stem}_{RUN_ID}.log"

    command = [
        'jupyter',
        'nbconvert',
        '--to',
        'notebook',
        '--execute',
        str(input_path.name),
        '--output',
        output_name,
        '--output-dir',
        str(EXECUTED_DIR),
        '--ExecutePreprocessor.timeout=-1',
    ]

    env = os.environ.copy()
    env['PYTHONPATH'] = str(SRC_DIR) + os.pathsep + env.get('PYTHONPATH', '')

    started_at = time.time()
    print(f'Running {notebook_name} ...')
    with open(log_path, 'w') as log_file:
        process = subprocess.run(
            command,
            cwd=SRC_DIR,
            env=env,
            stdout=log_file,
            stderr=subprocess.STDOUT,
            text=True,
        )

    elapsed_seconds = round(time.time() - started_at, 2)
    result = {
        'notebook': notebook_name,
        'status': 'success' if process.returncode == 0 else 'failed',
        'returncode': process.returncode,
        'elapsed_seconds': elapsed_seconds,
        'executed_notebook': str(output_path),
        'log': str(log_path),
    }

    if process.returncode == 0:
        print(f"Done {notebook_name} in {elapsed_seconds}s")
        print(f"Executed notebook: {output_path}")
    else:
        print(f"Failed {notebook_name} after {elapsed_seconds}s")
        print(f"Log: {log_path}")

    return result


In [ ]:
results = []

for notebook_name in RUN_NOTEBOOKS:
    result = execute_notebook(notebook_name)
    results.append(result)

    if result['status'] == 'failed' and STOP_ON_ERROR:
        break

summary_path = LOG_DIR / f'run_colab_summary_{RUN_ID}.json'
with open(summary_path, 'w') as f:
    json.dump(results, f, indent=2)

print('\nSummary:')
print(json.dumps(results, indent=2))
print('Summary file:', summary_path)

failed = [item for item in results if item['status'] == 'failed']
if failed:
    raise RuntimeError(f"{len(failed)} notebook(s) failed. Check logs under {LOG_DIR}.")
